# Results walkthrough - Taste, Trends & Algorithms

This notebook reads the files produced by `scripts/run_pipeline.py` and reproduces the key tables and charts. Run the pipeline first. The notebook itself never loads the 25M-row ratings file.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import visuals as V

MODE = "full"   # or "dev"
RES = ROOT / "reports" / "results" / MODE
print(RES, "exists:", RES.exists())

## 1. Dataset summary (Spark SQL output)

In [ ]:
summary = json.loads((RES / "eda" / "summary.json").read_text())
pd.Series({k: v for k, v in summary.items() if not isinstance(v, (dict, list))})

## 2. Popularity and the long tail

In [ ]:
V.long_tail(pd.read_csv(RES / "eda" / "long_tail_curve.csv"))

In [ ]:
pd.read_csv(RES / "eda" / "popularity_tiers.csv")

## 3. Genres

In [ ]:
V.genre_bars(pd.read_csv(RES / "eda" / "genre_stats.csv"))

## 4. Model evaluation: ALS vs popularity vs random

In [ ]:
metrics = json.loads((RES / "metrics.json").read_text())
print("RMSE:", metrics["rmse"])
comp = pd.read_csv(RES / "model_comparison.csv")
comp

In [ ]:
V.metric_comparison(comp, ["precision_at_10", "recall_at_10", "ndcg_at_10"], "Top-10 accuracy")

In [ ]:
V.tier_exposure(pd.read_csv(RES / "tier_exposure.csv"))

## 5. Recommend for one user from the exported factors (no Spark)

In [ ]:
from src.recommender_core import load_factor_model
from src.io_utils import read_table
proc = ROOT / "data" / "processed" / MODE
model = load_factor_model(proc)
cat = read_table(proc / "movie_catalogue").set_index("movieId")
user = int(model.user_ids[0])
recs = model.recommend(user, 10)   # note: pass exclude=<rated movie ids> to hide rated movies
recs.assign(title=cat.loc[recs.movieId, "title"].values)